# GAS-BayesSHAP — final paper evidence (Q1 gap closure)

This notebook runs the **remaining evidence** flagged open by the deep
audit of the Q1 package: SOTA-style baselines, N=50 finite-population
reruns, instrumented matched-budget curves, ablation N=20, regime
semantics N≥20, the frontier-completion probe (nominal 1−δ),
an adversarial stress test of the finite-population certificate, and
coverage calibration at tight ε.  It orchestrates the real CLI scripts
only (`scripts/run_sota_baselines.py`, `scripts/run_paper_experiments.py`,
`scripts/ablation.py`, `scripts/regime_semantics.py`,
`scripts/probe_width_tightness.py`, `scripts/stress_finite_population.py`,
`scripts/coverage_validation.py`); it duplicates no scientific algorithm.

## 0. Environment

In [ ]:
import sys, os, time, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import gas_bayesshap

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
print("GAS-BayesSHAP", gas_bayesshap.__version__)
wine_csv = ROOT / "data" / "winequality-white.csv"
air_csv  = ROOT / "data" / "Beijing_MultiSite_AirQuality.csv"
print("wine data:", "OK" if wine_csv.exists() else "MISSING (loader falls back to synthetic)")
print("air  data:", "OK" if air_csv.exists() else "MISSING (loader falls back to synthetic)")

## 0b. Configuration

In [ ]:
# --- sizes (env-overridable; see header for smoke defaults) ---
SOTA_N     = int(os.environ.get("SOTA_N", "20"))
FP_N       = int(os.environ.get("FP_N", "50"))
ABL_N      = int(os.environ.get("ABL_N", "20"))
REG_N      = int(os.environ.get("REG_N", "20"))
PROBE_BUDGETS = os.environ.get("PROBE_BUDGETS", "150000,200000")
STRESS_TRIALS = int(os.environ.get("STRESS_TRIALS", "200"))
COV_TRIALS = int(os.environ.get("COV_TRIALS", "200"))
SKIP       = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}
EPS        = float(os.environ.get("GAS_EPS", "0.05"))
BUDGET     = int(os.environ.get("GAS_BUDGET", "3000"))

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}")
        return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT)
    dt = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"SOTA_N={SOTA_N} FP_N={FP_N} ABL_N={ABL_N} REG_N={REG_N} "
      f"PROBE_BUDGETS={PROBE_BUDGETS} STRESS_TRIALS={STRESS_TRIALS} "
      f"COV_TRIALS={COV_TRIALS} SKIP={sorted(SKIP)}")

## A. SOTA-style baselines (OddSHAP + ShaplEIG, N=20)

`scripts/run_sota_baselines.py` compares GAS-BayesSHAP against
OddSHAP-style (exact Shapley of the log-odds game) and ShaplEIG-style
(GP posterior-mean Shapley, non-certified) at matched budgets
{256, 1024, 2048} on wine + air.  **Honest labelling:** official
OddSHAP/ShaplEIG code is not public in this environment; these are
method-style reimplementations, reported as non-certified references.

In [ ]:
run("run_sota_baselines.py", "--n", str(SOTA_N), tag=f"A. SOTA baselines N={SOTA_N}",
    skip="A" in SKIP)

## B. N=50 finite-population reruns (wine + air)

The committed N=50 tables use the spec range (width ~9.1–9.4).  This
reruns both datasets at the **same** standard budget with
`range_mode=finite_population` (Theorem E), producing
`paper_{wine,air}_n50_budget3000_rangefinite_population_summary.csv`.
**Honest expectation:** width ~1 (vs 9), same RMSE, same coverage;
sign-cert still 0 at K=3000 because the dominant attribution (~0.26)
is below the width — sign certification needs K ≥ 3×10^4 (Section F).

In [ ]:
run("run_paper_experiments.py", "--only", "wine", "--n", str(FP_N),
    "--eps", str(EPS), "--budget", str(BUDGET), "--range-mode", "finite_population",
    tag=f"B1. wine N={FP_N} finite-population", skip="B" in SKIP)

In [ ]:
run("run_paper_experiments.py", "--only", "air", "--n", str(FP_N),
    "--eps", str(EPS), "--budget", str(BUDGET), "--range-mode", "finite_population",
    tag=f"B2. air N={FP_N} finite-population", skip="B" in SKIP)

## C. Instrumented matched-budget curves

`run_curves` now records **actual** coalition/model call counts per
method (the audit's matched-budget fairness fix).  This reruns wine +
air at K ∈ {128, 256, 512, 1024, 2048}, N=8 (hardcoded in the runner),
spec range, and writes `paper_{wine,air}_matched_budget.csv` with the
new `*_evals_actual` columns.

In [ ]:
run("run_paper_experiments.py", "--only", "curves", "--range-mode", "spec",
    tag="C. instrumented matched-budget curves", skip="C" in SKIP)

## D. Ablation N=20 (regenerates tier-4 width/coverage)

`scripts/ablation.py --dataset wine --K 1000 --n 20` regenerates
`paper_ablation_wine_summary.csv` with the new per-instance tier-4
width / simultaneous coverage / sign-cert columns.  Expected: full
≈0.003 vs uniform ≈0.039 / neyman ≈0.007 / GP-only ≈0.071.

In [ ]:
run("ablation.py", "--dataset", "wine", "--K", "1000", "--n", str(ABL_N),
    tag=f"D. ablation wine N={ABL_N} K=1000", skip="D" in SKIP)

## E. Regime semantics N≥20

`scripts/regime_semantics.py --n 20` names the 4 air-quality regimes and
reports per-regime Spearman driver correlation with **N=20 instances**
(audit: the committed N=1–2 per regime is pilot-scale).  Writes
`paper_regime_semantics_summary.csv`.

In [ ]:
run("regime_semantics.py", "--n", str(REG_N), "--clusters", "4",
    "--eps", str(EPS), "--budget", str(BUDGET),
    tag=f"E. regime semantics N={REG_N}", skip="E" in SKIP)

## F. Frontier completion — fp probe at K = 150k / 200k

The wine probe showed sign certification (1 feature, validated vs exact)
at K ≥ 30k with realised level 0.372 at K=100k.  Corollary E predicts
the **nominal 1−δ** certificate when the coupon thresholds close at
K ≈ 2×10^5 for M=11.  This runs the fp probe at K = 150k / 200k to
confirm `certificate_at_nominal_level=True` (or report how close).
~5–8 min per budget on a laptop.

In [ ]:
run("probe_width_tightness.py", "--range-mode", "finite_population",
    "--budgets", PROBE_BUDGETS, tag=f"F. fp probe K={PROBE_BUDGETS}",
    skip="F" in SKIP)

## G. Adversarial stress test of the fp certificate

`scripts/stress_finite_population.py` plants a **rare extreme coalition**
in the game (the impossibility construction) and verifies the fp
accounting is honest: at M=3 (coupon closes fast) coverage must reach
the nominal level; at M=6 (extreme pair rare) the flag
`certificate_at_nominal_level=False` and the realised level must be
reported correctly.  Writes `paper_stress_finite_population.json`.

In [ ]:
run("stress_finite_population.py", "--trials", str(STRESS_TRIALS),
    tag=f"G. fp adversarial stress (R={STRESS_TRIALS})", skip="G" in SKIP)

## H. Coverage calibration at tight ε

The R=500 calibration used ε=1.5.  This runs `coverage_validation.py` at
ε ∈ {0.05, 0.1, 0.2, 0.5} (R=COV_TRIALS each) for the spec and fp modes,
persisting a JSON per (ε, mode).  At tight ε the budget often exhausts
before the width target — `finite_width_rate < 1` is the expected,
honest outcome (certificates valid but wide at low K).

In [ ]:
results = {}
for eps in (0.05, 0.1, 0.2, 0.5):
    for mode in ("spec", "finite_population"):
        if "H" in SKIP:
            continue
        cmd = [sys.executable, str(SCRIPTS / "coverage_validation.py"),
               "--trials", str(COV_TRIALS), "--M", "3", "--epsilon", str(eps),
               "--delta", "0.05", "--max-budget", "300", "--range-mode", mode]
        r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
        print(r.stdout)
        if r.returncode != 0:
            print(r.stderr); continue
        import re
        d = {}
        for line in r.stdout.splitlines():
            m = re.match(r"^\s*(.*?)\s*:\s*(.+?)\s*$", line)
            if m and m.group(1) in ("finite_width_rate", "empirical_coverage",
                                    "coverage_given_finite", "mean_width",
                                    "median_width", "max_width",
                                    "oracle_query_cost (mean)"):
                k = m.group(1).replace(" ", "_").replace("(", "").replace(")", "")
                try: d[k] = float(m.group(2)) if "." in m.group(2) else int(m.group(2))
                except ValueError: d[k] = m.group(2)
        results[f"eps{eps}_{mode}"] = d
        for dest in (ROOT / "results" / "paper_experiments", ROOT / "main_results"):
            dest.mkdir(parents=True, exist_ok=True)
            (dest / f"paper_calibration_eps{eps}_{mode}.json").write_text(
                json.dumps({"epsilon": eps, "mode": mode, "trials": COV_TRIALS, **d}, indent=1))
print("persisted paper_calibration_eps{0.05,0.1,0.2,0.5}_{spec,fp}.json")

## I. Summary of produced artifacts

In [ ]:
def show(name, path):
    p = Path(path)
    if not p.exists():
        print(f"[{name}] NOT FOUND: {p.name}"); return
    print(f"\n[{name}]")
    if p.suffix == ".json":
        print(json.dumps(json.loads(p.read_text()), indent=1)[:1200])
    else:
        print(pd.read_csv(p).to_string(index=False)[:1500])

show("A. SOTA baselines", ROOT / "main_results" / "paper_sota_baselines_comparison.csv")
show("B1. wine fp N=50", ROOT / "main_results" / f"paper_wine_n{FP_N}_budget{BUDGET}_rangefinite_population_summary.csv")
show("B2. air fp N=50", ROOT / "main_results" / f"paper_air_n{FP_N}_budget{BUDGET}_rangefinite_population_summary.csv")
show("C. wine curves", ROOT / "main_results" / "paper_wine_matched_budget.csv")
show("D. ablation", ROOT / "main_results" / "paper_ablation_wine_summary.csv")
show("E. regimes", ROOT / "main_results" / "paper_regime_semantics_summary.csv")
show("F. fp probe", ROOT / "main_results" / "paper_width_probe_finite_population.csv")
show("G. stress", ROOT / "main_results" / "paper_stress_finite_population.json")
for eps in (0.05, 0.1, 0.2, 0.5):
    for mode in ("spec", "finite_population"):
        show(f"H. calibration eps={eps} {mode}",
             ROOT / "main_results" / f"paper_calibration_eps{eps}_{mode}.json")

## Expected runtime (laptop) and honest notes
- **Full run ≈ 5–7 h:** A ≈ 40–60 min, B ≈ 2.5–3 h (wine+air N=50),
  C ≈ 1 h, D ≈ 35 min, E ≈ 15 min, F ≈ 10 min, G ≈ 10 min, H ≈ 15 min.
- **Smoke run (≈ 10 min):**
  `GAS_SKIP=C,D,E,H SOTA_N=2 FP_N=1 PROBE_BUDGETS=20000 STRESS_TRIALS=10 COV_TRIALS=10`
- **What each section can/cannot claim is stated in its markdown cell.**
  In particular: sign-cert at K=3000 is NOT expected (Section B);
  nominal 1−δ certification is the target of Section F (K≈2×10^5);
  Section G claims honesty of the flag/level, not coverage under an
  open coupon.